# 05 – Database Build

**Objective:** Assemble the master building table by joining cleaned building footprints with
district boundaries, computing area statistics, and writing the final GeoPackage and Parquet
outputs.

**Prerequisites:** Run notebooks 02 and 03 first to ensure the input GeoPackages exist.

In [ ]:
from __future__ import annotations

import geopandas as gpd
import pandas as pd
import yaml
from pathlib import Path
from shapely.validation import make_valid

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent
CONFIG_PATH  = PROJECT_ROOT / "config" / "config.yaml"

with CONFIG_PATH.open() as fh:
    cfg = yaml.safe_load(fh)

db = cfg["database"]
BUILDINGS_RAW    = PROJECT_ROOT / db["buildings_raw_path"]
BUILDINGS_CLEAN  = PROJECT_ROOT / db["buildings_clean_path"]
BUILDINGS_MASTER = PROJECT_ROOT / db["buildings_master_path"]
BUILDINGS_PARQ   = PROJECT_ROOT / db["buildings_parquet_path"]
DISTRICTS_PATH   = PROJECT_ROOT / db["districts_path"]

for p in [BUILDINGS_CLEAN, BUILDINGS_MASTER, BUILDINGS_PARQ]:
    p.parent.mkdir(parents=True, exist_ok=True)

PROJECT_CRS = cfg["crs"]["project_crs"]
print("Paths configured.")

## 1. Load raw buildings

In [ ]:
buildings = gpd.read_file(BUILDINGS_RAW, layer="buildings_raw")
print(f"Loaded {len(buildings):,} raw buildings  (CRS: {buildings.crs})")

## 2. Clean geometries

In [ ]:
n_invalid = (~buildings.geometry.is_valid).sum()
if n_invalid:
    print(f"Fixing {n_invalid} invalid geometries …")
    buildings["geometry"] = buildings.geometry.apply(make_valid)

# Drop duplicates and empty geometries
buildings = buildings[~buildings.geometry.is_empty].drop_duplicates(subset=["geometry"])
buildings["area_m2"] = buildings.geometry.area.round(2)

buildings.to_file(BUILDINGS_CLEAN, driver="GPKG", layer="buildings_clean")
print(f"{len(buildings):,} clean buildings written to {BUILDINGS_CLEAN}")

## 3. Spatial join with districts

In [ ]:
districts = gpd.read_file(DISTRICTS_PATH, layer="gdynia_districts")
print(f"Loaded {len(districts)} districts")

# Use representative point for the join to handle edge cases
buildings_pts = buildings.copy()
buildings_pts["geometry"] = buildings.geometry.representative_point()

joined = gpd.sjoin(buildings_pts, districts[["geometry", "district_id", "district_name"]],
                   how="left", predicate="within")

# Restore original polygon geometry
master = buildings.copy()
master["district_id"]   = joined["district_id"].values
master["district_name"] = joined["district_name"].values
master["source_confirmed"] = True

unmatched = master["district_id"].isna().sum()
print(f"Buildings with district: {len(master) - unmatched:,}  |  unmatched: {unmatched}")

## 4. Save master table

In [ ]:
master.to_file(BUILDINGS_MASTER, driver="GPKG", layer="buildings_master")
master.drop(columns="geometry").to_parquet(BUILDINGS_PARQ, index=False)
print(f"Master table: {len(master):,} rows")
print(f"  GeoPackage → {BUILDINGS_MASTER}")
print(f"  Parquet    → {BUILDINGS_PARQ}")